In [57]:
import os
from PIL import Image
import numpy as np
from sklearn.model_selection import KFold

def przygotuj_dane_dla_algorytmu(folder_zdrowe, folder_chore, rozmiar=(32, 32)):
    X_lista = []
    y_lista = []
    
    for plik in os.listdir(folder_zdrowe):
        if plik.endswith(".JPG") or plik.endswith(".png") or plik.endswith(".jpg"):
            sciezka = os.path.join(folder_zdrowe, plik)
            img = Image.open(sciezka).resize(rozmiar)
            piksele = np.asarray(img).flatten() / 255.0
            
            piksele_z_jedynka = np.insert(piksele, 0, 1.0)
            
            X_lista.append(piksele_z_jedynka)
            y_lista.append(0)
            
    for plik in os.listdir(folder_chore):
        if plik.endswith(".JPG") or plik.endswith(".png") or plik.endswith(".jpg"):
            sciezka = os.path.join(folder_chore, plik)
            img = Image.open(sciezka).resize(rozmiar)
            piksele = np.asarray(img).flatten() / 255.0
            
            piksele_z_jedynka = np.insert(piksele, 0, 1.0)
            
            X_lista.append(piksele_z_jedynka)
            y_lista.append(1)
            
    X_macierz = np.matrix(X_lista)
    y_macierz = np.matrix(y_lista).reshape(-1, 1)
    
    return X_macierz, y_macierz

In [58]:
# Funkcja regresji logistcznej
def h(theta, X):
    return 1.0/(1.0 + np.exp(-X * theta))

# Funkcja kosztu dla regresji logistycznej
def J(h, theta, X, y):
    m = len(y)
    h_val = h(theta, X)
    s1 = np.multiply(y, np.log(h_val))
    s2 = np.multiply((1 - y), np.log(1 - h_val))
    return -np.sum(s1 + s2, axis=0) / m

# Gradient dla regresji logistycznej
def dJ(h, theta, X, y):
    return 1.0 / len(y) * (X.T * (h(theta, X) - y))

# Metoda gradientu prostego dla regresji logistycznej
def GD(h, fJ, fdJ, theta, X, y, alpha=0.01, eps=10**-3, maxSteps=10000):
    errorCurr = fJ(h, theta, X, y)
    errors = [[errorCurr, theta]]
    while True:
        # oblicz nowe theta
        theta = theta - alpha * fdJ(h, theta, X, y)
        # raportuj poziom błędu
        errorCurr, errorPrev = fJ(h, theta, X, y), errorCurr
        # kryteria stopu
        if abs(errorPrev - errorCurr) <= eps:
            break
        if len(errors) > maxSteps:
            break
        errors.append([errorCurr, theta]) 
    return theta, errors

def classify(theta, X):
    prawdopodobienstwa = h(theta, X)
    # 1 (chory), 0 (zdrowy)
    return np.where(prawdopodobienstwa >= 0.5, 1, 0)

def ocena_jakosci(y_prawdziwe, y_przewidziane):
    y_praw = np.asarray(y_prawdziwe).flatten()
    y_przew = np.asarray(y_przewidziane).flatten()
    
    TP = np.sum((y_przew == 1) & (y_praw == 1))
    TN = np.sum((y_przew == 0) & (y_praw == 0))
    FP = np.sum((y_przew == 1) & (y_praw == 0))
    FN = np.sum((y_przew == 0) & (y_praw == 1))
    
    accuracy = (TP + TN) / len(y_praw)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return accuracy, precision, recall, f1

In [59]:
def walidacje_krzyzowa(X_train, y_train, k=5, alpha=0.01, eps=10**-5, maxSteps=500):
    print(f"\nwalidacja krzyżowa")
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    liczba_cech = X_train.shape[1]
    numer = 1
    
    acc_lista, f1_lista = [], []
    
    for train_idx, val_idx in kf.split(X_train):        
        # zbior trening i walidacja
        X_train_kf, X_val_kf = X_train[train_idx], X_train[val_idx]
        y_train_kf, y_val_kf = y_train[train_idx], y_train[val_idx]
        
        theta_start = np.matrix(np.zeros((liczba_cech, 1)))
        
        theta_kf, _ = GD(h, J, dJ, theta_start, X_train_kf, y_train_kf, 
                         alpha=alpha, eps=eps, maxSteps=maxSteps)
        
        y_przew_kf = classify(theta_kf, X_val_kf)
        acc_kf, prec_kf, rec_kf, f1_kf = ocena_jakosci(y_val_kf, y_przew_kf)
        
        print(f"Accuracy: {acc_kf:.4f}, F1: {f1_kf:.4f}")
        acc_lista.append(acc_kf)
        f1_lista.append(f1_kf)
        numer += 1

In [60]:
# POMIDORY
# ZBIÓR TRENINGOWY
folder_trening_zdrowe = "../plants_leafs/train/tomato_healthy"
folder_trening_chore = "../plants_leafs/train/tomato_bacterial_spots"
X_train, y_train = przygotuj_dane_dla_algorytmu(folder_trening_zdrowe, folder_trening_chore)

liczba_cech = X_train.shape[1]

walidacje_krzyzowa(X_train, y_train, k=5, alpha=0.01, eps=10**-5, maxSteps=500)

theta_start_final = np.matrix(np.zeros((liczba_cech, 1)))
thetaBest, _ = GD(h, J, dJ, theta_start_final, X_train, y_train, alpha=0.01, eps=10**-5, maxSteps=1000)

folder_test_zdrowe = "../plants_leafs/test/tomato_healthy"
folder_test_chore = "../plants_leafs/test/tomato_bacterial_spots"
X_test, y_test = przygotuj_dane_dla_algorytmu(folder_test_zdrowe, folder_test_chore)


y_przewidziane_test = classify(thetaBest, X_test)
acc, prec, rec, f1 = ocena_jakosci(y_test, y_przewidziane_test)

print(f"\nPOMIDORY")
print(f"Accuracy : {acc:.4f}")
print(f"Precision :  {prec:.4f}")
print(f"Recall :{rec:.4f}")
print(f"F1 Score :{f1:.4f}")


walidacja krzyżowa
Accuracy: 1.0000, F1: 1.0000
Accuracy: 0.7500, F1: 0.6667
Accuracy: 0.2500, F1: 0.4000
Accuracy: 1.0000, F1: 1.0000
Accuracy: 0.5000, F1: 0.6667

POMIDORY
Accuracy : 0.7500
Precision :  1.0000
Recall :0.5000
F1 Score :0.6667


In [61]:
# WINOGRONO
# ZBIÓR TRENINGOWY
folder_trening_zdrowe = "../plants_leafs/train/grape_healthy"
folder_trening_chore = "../plants_leafs/train/grape_rot"
X_train, y_train = przygotuj_dane_dla_algorytmu(folder_trening_zdrowe, folder_trening_chore)

liczba_cech = X_train.shape[1]

walidacje_krzyzowa(X_train, y_train, k=5, alpha=0.01, eps=10**-5, maxSteps=500)

theta_start_final = np.matrix(np.zeros((liczba_cech, 1)))
thetaBest, _ = GD(h, J, dJ, theta_start_final, X_train, y_train, alpha=0.01, eps=10**-5, maxSteps=1000)

folder_test_zdrowe = "../plants_leafs/test/grape_healthy"
folder_test_chore = "../plants_leafs/test/grape_rot"
X_test, y_test = przygotuj_dane_dla_algorytmu(folder_test_zdrowe, folder_test_chore)


y_przewidziane_test = classify(thetaBest, X_test)
acc, prec, rec, f1 = ocena_jakosci(y_test, y_przewidziane_test)

print(f"\nWINOGRONO")
print(f"Accuracy : {acc:.4f}")
print(f"Precision :  {prec:.4f}")
print(f"Recall :{rec:.4f}")
print(f"F1 Score :{f1:.4f}")


walidacja krzyżowa
Accuracy: 0.5000, F1: 0.5000
Accuracy: 1.0000, F1: 1.0000
Accuracy: 1.0000, F1: 1.0000
Accuracy: 0.5000, F1: 0.0000
Accuracy: 0.7500, F1: 0.8000

WINOGRONO
Accuracy : 1.0000
Precision :  1.0000
Recall :1.0000
F1 Score :1.0000


In [62]:
# JABŁKO
# ZBIÓR TRENINGOWY
folder_trening_zdrowe = "../plants_leafs/train/apple_healthy"
folder_trening_chore = "../plants_leafs/train/apple_rot"
X_train, y_train = przygotuj_dane_dla_algorytmu(folder_trening_zdrowe, folder_trening_chore)

liczba_cech = X_train.shape[1]

walidacje_krzyzowa(X_train, y_train, k=5, alpha=0.01, eps=10**-5, maxSteps=500)

theta_start_final = np.matrix(np.zeros((liczba_cech, 1)))
thetaBest, _ = GD(h, J, dJ, theta_start_final, X_train, y_train, alpha=0.01, eps=10**-5, maxSteps=1000)

folder_test_zdrowe = "../plants_leafs/test/apple_healthy"
folder_test_chore = "../plants_leafs/test/apple_rot"
X_test, y_test = przygotuj_dane_dla_algorytmu(folder_test_zdrowe, folder_test_chore)

y_przewidziane_test = classify(thetaBest, X_test)
acc, prec, rec, f1 = ocena_jakosci(y_test, y_przewidziane_test)

print(f"\nJABŁKO")
print(f"Accuracy : {acc:.4f}")
print(f"Precision :  {prec:.4f}")
print(f"Recall :{rec:.4f}")
print(f"F1 Score :{f1:.4f}")


walidacja krzyżowa
Accuracy: 1.0000, F1: 1.0000
Accuracy: 1.0000, F1: 1.0000
Accuracy: 0.5000, F1: 0.6667
Accuracy: 0.7500, F1: 0.6667
Accuracy: 0.5000, F1: 0.0000

JABŁKO
Accuracy : 0.7500
Precision :  0.6667
Recall :1.0000
F1 Score :0.8000
